# Files

A tournament scoreboard disappears when its program stops. Your challenge is to save scores in a file, load them into useful Python values, and find a player in a saved roster.

Every example writes its own known data before reading it, so you can run the notebook again safely.

## Lesson 1: Save Scores, Then Load Them Back

The tournament director wants the scoreboard to survive after the program stops. We will write one score per line, then read the whole saved file back as text.

### A safe save-and-load tool

`open(path, "w")` opens a file for writing. The `with` statement closes the file when its indented block ends. A file stores text, so the f-string in `f.write(f"{score}\n")` turns each score into text and adds a newline.

`\n` is the special **new line** character inside a string.
`print` turns each `\n` into a line break, so printing a whole file that already ends in `\n` shows a blank line at the end; that is why worked samples show the raw `"45\n70\n55\n"` form and why you compare it with `==` to check it.

For this first lesson, `f.read()` loads the whole file as one text string. Line-by-line reading begins in Lesson 2.

In [ ]:
# Save every score on its own line.
def save_scores(scores, path):
    with open(path, "w") as f:
        for score in scores:
            f.write(f"{score}\n")
    return path

def load_score_text(path):
    with open(path, "r") as f:
        text = f.read()
    return text

def score_text_round_trip(scores, path):
    save_scores(scores, path)
    return load_score_text(path)

print(score_text_round_trip([840, 920, 775], "lesson_l1_scores.txt"))
print(score_text_round_trip([610, 990], "lesson_l1_scores.txt"))

The first call saves three lines and returns the text `840\n920\n775\n`. The second call replaces that file with two known lines before reading it again. Mode `"w"` means **write a fresh file**. Mode `"r"` means **read an existing file**.

### Why use `with`?

Here is the old way in function form. The programmer has to remember a separate `close()` call.

In [ ]:
def save_one_score_old_way(score, path):
    f = open(path, "w")
    f.write(f"{score}\n")
    f.close()
    return path

print(save_one_score_old_way(730, "lesson_l1_old_way.txt"))

The worked call prints `lesson_l1_old_way.txt` because that path is the function's result, and it leaves `730` on one line in the file. The better form is `with open(path, "w") as f:`. **`with` closes the file for you automatically, even if your code crashes or you forget `close()`.** We will use `with` for every remaining file operation.

## Lesson 2: Turn Saved Lines into Score Statistics

The scoreboard file is readable, but a statistics tool cannot add text such as `"840"`. It needs to clean each saved line and turn it into an integer. Then it can update a statistic immediately or rebuild a list for Python's built-in tools.

### Read one line at a time

`for line in f:` visits each line in order. `line.strip()` removes its newline, and `int(...)` changes the cleaned text into an integer. The append step transforms each file line into one list item.

In [ ]:
def save_number_lines(scores, path):
    with open(path, "w") as f:
        for score in scores:
            f.write(f"{score}\n")
    return path

def load_number_lines(path):
    loaded_scores = []
    with open(path, "r") as f:
        for line in f:
            score = int(line.strip())
            loaded_scores.append(score)
    return loaded_scores

def number_line_round_trip(scores, path):
    save_number_lines(scores, path)
    return load_number_lines(path)

print(number_line_round_trip([14, 21, 18], "lesson_l2_numbers.txt"))
print(number_line_round_trip([32, 27], "lesson_l2_numbers.txt"))

### Calculate statistics while reading the file

The next tools do not wait for a separate list before using the loop patterns. Each one saves known scores, opens that file for reading, changes the line to an integer, and immediately updates its answer. A running total keeps an accumulator named `total`. A count-by-condition changes `count` only when a saved score meets the condition. An explicit best-so-far scan changes `best` only when a larger saved score appears.

For comparison, the built-in statistics tool rebuilds a list first, then calls `min`, `max`, `sum`, and `len` on that list.

In [ ]:
def running_total_from_file(scores, path):
    save_number_lines(scores, path)
    total = 0
    with open(path, "r") as f:
        for line in f:
            score = int(line.strip())
            total = total + score
    return total

def count_from_file(scores, target, path):
    save_number_lines(scores, path)
    count = 0
    with open(path, "r") as f:
        for line in f:
            score = int(line.strip())
            if score >= target:
                count = count + 1
    return count

def built_in_stats_from_file(scores, path):
    save_number_lines(scores, path)
    loaded_scores = load_number_lines(path)
    return [min(loaded_scores), max(loaded_scores), sum(loaded_scores), len(loaded_scores)]

def best_score_from_file(scores, path):
    save_number_lines(scores, path)
    best = scores[0]
    with open(path, "r") as f:
        for line in f:
            score = int(line.strip())
            if score > best:
                best = score
    return best

print(running_total_from_file([420, 680, 510, 735], "lesson_l2_total.txt"))
print(count_from_file([420, 680, 510, 735], 600, "lesson_l2_count.txt"))
print(built_in_stats_from_file([420, 680, 510, 735], "lesson_l2_builtins.txt"))
print(best_score_from_file([420, 680, 510, 735], "lesson_l2_best.txt"))

The worked calls report a total of `2345`, two scores at least `600`, the list `[420, 735, 2345, 4]`, and a best-so-far result of `735`. Each call writes its own file before reading it. The last function deliberately uses a direct file loop instead of `max` so you can see the find-extreme pattern transfer to saved lines.

## Lesson 3: Save Records and Search a Roster

A coach needs more than a bare score. One saved player record must keep a name, a level, and points together. The coach also needs a search tool that checks roster lines in order and returns as soon as it finds the requested player.

### One record across several lines

This simple format uses three lines for one record: name first, then level, then points. The loader cleans every line. It keeps the name as text and transforms the remaining fields into integers before appending them to one record list.

`f.readline()` reads just the NEXT line and moves on, so a following `for line in f:` continues from the line after it.

In [ ]:
def save_player_record(name, level, points, path):
    with open(path, "w") as f:
        f.write(f"{name}\n")
        f.write(f"{level}\n")
        f.write(f"{points}\n")
    return path

def load_player_record(path):
    record = []
    with open(path, "r") as f:
        name = f.readline().strip()
        record.append(name)
        for line in f:
            number = int(line.strip())
            record.append(number)
    return record

def player_record_round_trip(name, level, points, path):
    save_player_record(name, level, points, path)
    return load_player_record(path)

print(player_record_round_trip("Mina", 4, 860, "lesson_l3_record.txt"))
print(player_record_round_trip("Omar", 6, 1040, "lesson_l3_record.txt"))

### Search for the first matching line

A linear search checks one line, then the next. Equality with `==` asks whether the cleaned line matches the target exactly. A matching line returns immediately. The not-found return belongs after the loop because the function must check every line before giving up.

In [ ]:
def save_roster(names, path):
    with open(path, "w") as f:
        for name in names:
            f.write(f"{name}\n")
    return path

def find_player(target, path):
    with open(path, "r") as f:
        for line in f:
            if line.strip() == target:
                return f"Found {target}."
    return f"{target} was not found."

def roster_search(names, target, path):
    save_roster(names, path)
    return find_player(target, path)

print(roster_search(["Ari", "Bo", "Chen"], "Bo", "lesson_l3_roster.txt"))
print(roster_search(["Ari", "Bo", "Chen"], "Devi", "lesson_l3_roster.txt"))

The first search returns `Found Bo.` from inside the loop. The second checks all three lines, then reaches the fallback after the loop and returns `Devi was not found.`